# Model 5: Open-Set Score Fusion (OSSF)

Baseline: Dalvin Model 3b. Extension: final-stage logistic score fusion for ranking `new_individual` against known identities.

The experiment compares three methods on the same candidates, trained stack and conditional identity ordering: the legacy probability rule, the existing detector alone, and OSSF. Earlier stacking stages retain the legacy rule in every method. Only the final unknown probability changes.

**Recorded validation:** 506 queries; legacy rule MAP@5 0.916206, detector-only 0.909947, OSSF 0.911594. Delta is -0.0046113306982873414. The extension improves known-query performance slightly but loses more on unseen animals. No overall gain or leaderboard result is claimed. The table is a historical result; executed cells below produce the current run's result.

## Model and evaluation design

MIEW-ID normalization → per-species PCA/LDA retrieval → RootSIFT/RANSAC evidence → candidate feature tables → three sequential LightGBM stacking stages → logistic unknown-score fusion → ranked top-five labels.

Training identities are split into non-overlapping stage groups (60% total), calibration (20%) and outer validation (20%). Base models are frozen before calibration; fusion is frozen before evaluation. Each episode contains a labelled gallery, known queries and held-out unknown identities. PCA and query-neighbour features may use unlabelled episode queries; LDA uses gallery labels. Query truth is excluded from predictor features.

The legacy score is `(p_detector + clip(1-sum(candidate_binary_scores), 0, 1))/2`. OSSF uses imputation, scaling and logistic regression on detector/legacy scores, candidate uncertainty, raw/LDA margins, SIFT support, viewpoint availability, gallery support and species. Known identities receive `(1-p_new)*normalized_candidate_score`; `new_individual` receives `p_new`. Candidate scores are learned ranking evidence, not guaranteed calibrated probabilities.

The independent sequential-stage protocol differs from the team's original full-data simulation stacking. The local baseline is not a reproduction of its logged public leaderboard score. This previously examined evaluation cannot serve as untouched evidence for subsequent tuning.

Execution target: Kaggle CPU; competition input is projectwild3.


In [ ]:
# If imports are missing, enable Internet temporarily and uncomment this line.
# %pip install "numpy>=1.26,<2.5" "pandas>=2.1,<3" "scipy>=1.11,<2" "scikit-learn>=1.4,<2" "lightgbm>=4.3,<5" "opencv-python-headless>=4.8,<5" "joblib>=1.3,<2"


## 1. Experiment source snapshot
The following cell writes the embedded, versioned baseline and OSSF implementation. It also saves method documentation and recorded validation analysis beside the executable source.


In [ ]:
from pathlib import Path
SCRIPT_DIR = Path("/kaggle/working/abhay_code")
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
(SCRIPT_DIR / 'dalvin_core.py').write_text('"""Adapted Dalvin Model 3b functions. See README.md for attribution and changes."""\nimport os\n\nimport re\n\nfrom multiprocessing import Pool\n\nimport cv2\n\nimport lightgbm as lgb\n\nimport numpy as np\n\nimport pandas as pd\n\nfrom scipy import sparse\n\nfrom scipy.sparse.csgraph import connected_components\n\nfrom sklearn.decomposition import PCA\n\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\n\n\n\n\ndef unit(a):\n    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)\n\ndef embed(gallery, labels, queries):\n    pca = PCA(min(384, gallery.shape[1], len(gallery) + len(queries) - 1), random_state=0).fit(np.r_[gallery, queries])      # unsupervised, sees the queries too\n    lda = LDA(solver="eigen", shrinkage=0.3).fit(pca.transform(gallery), labels)\n    project = lambda a: unit(lda.transform(pca.transform(a)))\n    return project(gallery), project(queries)\n\ndef per_identity(sim, inv, n):\n    """Per query: max and mean-of-top-2 similarity to each identity\'s gallery images."""\n    best, top2 = np.empty((sim.shape[0], n)), np.empty((sim.shape[0], n))\n    for j in range(n):\n        s = sim[:, inv == j]\n        best[:, j], top2[:, j] = s.max(1), np.sort(s, 1)[:, -2:].mean(1)\n    return best, top2\n\ndef species_features(gi, qi, sp):\n    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""\n    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq = embed(Rg, yg, Rq)\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    support = (inl_g >= 8).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n    # transductive: neighbours among the queries themselves\n    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)\n    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]\n    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]\n    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],\n    })\n    return q, cd\n\ndef build_frames(gi, qi):\n    qs, cs, order, off = [], [], [], 0\n    for sp in ("hyena", "leopard"):\n        g, qq = gi[SP[gi] == sp], qi[SP[qi] == sp]\n        q, c = species_features(g, qq, sp)\n        c["q"] += off\n        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off\n        qs.append(q), cs.append(c), order.append(qq)\n        off += len(qq)\n    return dict(q=pd.concat(qs, ignore_index=True), c=pd.concat(cs, ignore_index=True), order=np.concatenate(order))\n\ndef duplicate_groups(ix, link):\n    """Group images of one identity that are near-duplicates (same encounter); each image alone if link is None."""\n    if link is None:\n        return np.arange(len(ix))\n    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])\n    return connected_components(sparse.csr_matrix(edges))[1]\n\n\ndef columns(s):\n    qf = [c for c in s["q"].columns if c != "unk" and not INDEX_COLS.match(c)]\n    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]\n    return qf, cf\n\ndef candidate_matrix(s, qf, cf):\n    return pd.concat([s["c"][cf].reset_index(drop=True), s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)\n\ndef fit(sims):\n    """new-individual detector, closed-set candidate scorer, and all-query candidate scorer."""\n    qf, cf = columns(sims[0])\n    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0\n    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])\n    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])\n    ck = np.concatenate([known(s) for s in sims])\n    detector = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": min(300, PARAMS["n_estimators"]), "num_leaves": 15}).fit(\n        pd.concat([s["q"][qf] for s in sims]), np.concatenate([s["q"].unk.to_numpy() for s in sims]))\n    return detector, lgb.LGBMClassifier(**PARAMS).fit(cx[ck], cy[ck]), lgb.LGBMClassifier(**PARAMS).fit(cx, cy), (qf, cf)\n\ndef predict(models, s):\n    detector, closed, anyq, (qf, cf) = models\n    nq, cx = len(s["q"]), candidate_matrix(s, qf, cf)\n    p_new = detector.predict_proba(s["q"][qf])[:, 1]\n    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)\n    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)\n    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)\n    return dict(pn=p_new, raw=raw, pc=within, pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)\n\ndef rank(s, r):\n    """Stable top-five; ties preserve the conditional known-identity ordering."""\n    nq = len(s["q"])\n    known = s["c"].ident.to_numpy(dtype=object).reshape(nq, K)\n    order = np.argsort(-r["pc"], axis=1, kind="stable")\n    labels = np.concatenate([np.take_along_axis(known, order, 1), np.full((nq, 1), "new_individual", dtype=object)], 1)\n    weights = np.take_along_axis((1-r["pb"])[:, None]*r["pc"], order, 1)\n    scores = np.concatenate([weights, r["pb"][:, None]], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, axis=1, kind="stable")[:, :5], 1)\n\ndef map5(preds, truth):\n    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0) for row, t in zip(preds, truth)])\n\ndef augment(s, r):\n    """Add the previous round\'s predictions for each query and for its nearest query neighbours."""\n    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])\n    codes = pd.factorize(c.ident)[0].reshape(nq, K)\n    q["r_pn"], q["r_pb"], q["r_rawsum"], q["r_rawmax"] = r["pn"], r["pb"], r["raw"].sum(1), r["raw"].max(1)\n    c["r_raw"], c["r_pc"] = r["raw"].ravel(), r["pc"].ravel()\n    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):     # embedding / SIFT neighbours\n        evidence, support, pb, w, top = np.zeros((nq, K)), np.zeros((nq, K)), [], [], []\n        for k in range(5):\n            idx, sim = q[f"{kind}{k}"].to_numpy(), q[f"{weight}{k}"].to_numpy() / scale\n            wk = np.clip(sim - 0.4, 0, None) if kind == "nb" else np.clip(sim, 0, 1)\n            agree = ((codes[idx][:, None, :] == codes[:, :, None]) * r["raw"][idx][:, None, :]).sum(2)\n            evidence = np.maximum(evidence, np.minimum(agree, np.clip(sim, 0, 1)[:, None]))\n            support += wk[:, None] * agree\n            pb.append(r["pb"][idx]), w.append(wk), top.append(r["raw"][idx].max(1))\n        pb, w = np.stack(pb, 1), np.stack(w, 1)\n        q[f"{kind}_pb0"], q[f"{kind}_pbw"] = pb[:, 0], (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)\n        q[f"{kind}_pbmin"], q[f"{kind}_w"], q[f"{kind}_rm0"] = np.where(w > 0, pb, 1).min(1), w.sum(1), top[0]\n        c[f"r_{kind}ev"], c[f"r_{kind}sum"] = evidence.ravel(), support.ravel()\n    return dict(s, q=q, c=c)\n\nK = 15\nNEIGHBOURS = 150\nSIFT_PULL = 0.05\nVIEWS = ["left", "right", "front", "back", "frontleft", "frontright", "backleft", "backright", "up", "down"]\nPARAMS = dict(objective="binary", learning_rate=.05, num_leaves=31, min_child_samples=30, subsample=.8, subsample_freq=1, colsample_bytree=.8, n_estimators=400, verbose=-1, random_state=42, n_jobs=2)\nINDEX_COLS = re.compile(r"^(nb|nw|sb|sw)\\d$")\nREGIMES = [None, (0.95, 100), (0.99, 80), (0.9, 60)]\n', encoding="utf-8")
(SCRIPT_DIR / 'train_eval.py').write_text('"""Model 5: Open-Set Score Fusion (OSSF), based on Dalvin Model 3b.\n\nRun --help. Validation is identity-disjoint from every supervised training stage.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport platform\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfrom functools import lru_cache\nfrom threading import Lock\n\nimport cv2\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nimport dalvin_core as b\n\nMODEL_NAME = "Model 5: Open-Set Score Fusion (OSSF)"\nMODEL_VERSION = "1.0"\nNEW = "new_individual"\n\n\ndef find_sift(root, split):\n    """Kaggle mounts SIFT beside the metadata folder, sometimes as extracted NPZ directories."""\n    name = f"sift_features_{split}.npz"\n    for parent in (root, root.parent, root.parent.parent):\n        candidate = parent / name\n        if candidate.exists():\n            return candidate\n    raise FileNotFoundError(f"Cannot find {name} in {root} or its two parent directories. "\n                            "Attach the full projectwild3 competition input, including both SIFT folders.")\n\n\nclass SiftSource:\n    def __init__(self, path):\n        self.archive = None\n        self.files = {}\n        if path.is_file():\n            self.archive = np.load(path)\n        else:\n            for file in path.rglob("*.npy"):\n                if file.stem in self.files:\n                    raise ValueError(f"Duplicate SIFT key {file.stem} in {path}")\n                self.files[file.stem] = file\n            if not self.files:\n                raise ValueError(f"No extracted .npy SIFT arrays in {path}")\n\n    def __getitem__(self, key):\n        return self.archive[key] if self.archive is not None else np.load(self.files[key], allow_pickle=False)\n\n\ndef load_data(root):\n    train = pd.read_csv(root / "train.csv")\n    test = pd.read_csv(root / "test.csv")\n    ids = pd.read_csv(root / "miewid_ids.csv")\n    all_df = pd.concat([train, test.assign(identity=None)], ignore_index=True)\n    if not all_df.id.is_unique or not ids.id.is_unique:\n        raise ValueError("Duplicate image IDs")\n    for col in ("species", "viewpoint", "identity"):\n        if col not in all_df:\n            raise ValueError(f"Missing column: {col}")\n    if train.identity.isna().any() or not set(all_df.species) <= {"hyena", "leopard"}:\n        raise ValueError("Invalid identity/species metadata")\n    raw = np.load(root / "miewid_features.npy", mmap_mode="r")\n    if len(raw) != len(ids) or set(ids.id) != set(all_df.id):\n        raise ValueError("Feature ID mapping does not match metadata")\n    ix = pd.Index(ids.id).get_indexer(all_df.id)\n    X = np.asarray(raw[ix], dtype=np.float32)\n    if not np.isfinite(X).all() or np.any(np.linalg.norm(X, axis=1) == 0):\n        raise ValueError("Features contain nonfinite values or zero vectors")\n    b.X = b.unit(X)\n    b.IDS = all_df.id.to_numpy()\n    b.Y = all_df.identity.to_numpy(dtype=object)\n    b.SP = all_df.species.to_numpy()\n    b.VIEW = all_df.viewpoint.map({v: i for i, v in enumerate(b.VIEWS)}).fillna(-1).to_numpy(int)\n    return train, test, all_df\n\n\nclass SiftCache:\n    """Role-local neighbour selection; thread workers share lazily loaded descriptors."""\n    def __init__(self, root, out, workers, neighbours):\n        paths = [find_sift(root, s) for s in ("train", "test")]\n        print("SIFT sources:", *paths, sep="\\n", flush=True)\n        self.z = [SiftSource(path) for path in paths]\n        self.read_lock = Lock()\n        self.workers, self.neighbours = workers, neighbours\n        fingerprint = hashlib.sha256(b.IDS.tobytes() if b.IDS.dtype != object else "\\n".join(b.IDS).encode())\n        fingerprint.update(np.ascontiguousarray(b.X).tobytes())\n        self.path = out / ("sift_" + fingerprint.hexdigest()[:16] + ".npz")\n        self.done = {}\n        if self.path.exists():\n            with np.load(self.path) as z:\n                self.done = {tuple(p): tuple(v) for p, v in zip(z["pairs"], z["values"])}\n\n    @lru_cache(maxsize=1200)\n    def descriptors(self, i):\n        iid = b.IDS[i]\n        z = self.z[0 if iid.startswith("tr_") else 1]\n        # NpzFile\'s shared ZIP handle is not safe for concurrent reads.\n        with self.read_lock:\n            d = np.asarray(z[f"{iid}_desc"], dtype=np.float32)\n            xy = np.asarray(z[f"{iid}_xy"], dtype=np.float32)\n        if d.ndim != 2 or xy.shape != (len(d), 2):\n            raise ValueError(f"Malformed SIFT arrays for {iid}")\n        d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1))\n        return d, xy\n\n    def verify(self, pair):\n        a, c = pair\n        da, xa = self.descriptors(a)\n        dc, xc = self.descriptors(c)\n        if min(len(da), len(dc)) < 4:\n            return 0, 0\n        matches = cv2.BFMatcher().knnMatch(da, dc, k=2)\n        good = [m[0] for m in matches if len(m) == 2 and m[0].distance < .8*m[1].distance]\n        if len(good) < 4:\n            return len(good), 0\n        # Local RNG seed per task. OpenCV RNG state is thread-local.\n        cv2.setRNGSeed(int((int(a)*100003 + int(c)) % (2**31-1)))\n        _, mask = cv2.estimateAffinePartial2D(\n            np.float32([xa[m.queryIdx] for m in good]),\n            np.float32([xc[m.trainIdx] for m in good]),\n            method=cv2.RANSAC, ransacReprojThreshold=10.)\n        return len(good), int(mask.sum()) if mask is not None else 0\n\n    def activate(self, indices):\n        wanted = set()\n        for sp in ("hyena", "leopard"):\n            ix = indices[b.SP[indices] == sp]\n            # Chunked similarities avoid allocating the full role x role matrix.\n            for start in range(0, len(ix), 256):\n                q = ix[start:start+256]\n                s = b.X[q] @ b.X[ix].T\n                s[np.arange(len(q)), np.arange(start, start+len(q))] = -np.inf\n                k = min(self.neighbours, len(ix)-1)\n                nb = np.argsort(-s, axis=1)[:, :k]\n                for a, row in zip(q, nb):\n                    wanted.update((min(int(a), int(c)), max(int(a), int(c))) for c in ix[row])\n        missing = sorted(wanted - self.done.keys())\n        print(f"SIFT: {len(wanted)} role-local pairs; {len(missing)} uncached", flush=True)\n        with ThreadPoolExecutor(max_workers=self.workers) as pool:\n            for j, (pair, value) in enumerate(zip(missing, pool.map(self.verify, missing)), 1):\n                self.done[pair] = value\n                if j % 10000 == 0:\n                    print(f"  verified {j}/{len(missing)}", flush=True)\n        pairs = sorted(self.done)\n        tmp = self.path.with_suffix(".tmp.npz")\n        np.savez_compressed(tmp, pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n                            values=np.array([self.done[p] for p in pairs]).reshape(-1, 2))\n        os.replace(tmp, self.path)\n        # Only expose requested pairs, even when cache contains other roles/test.\n        pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n        vv = np.array([self.done[tuple(p)] for p in pp]).reshape(-1, 2)\n        n = len(b.X)\n        def matrix(v):\n            a, c = pp.T\n            return sparse.coo_matrix((np.r_[v, v], (np.r_[a, c], np.r_[c, a])), shape=(n, n)).tocsr()\n        b.GOOD, b.INL = matrix(vv[:, 0]), matrix(vv[:, 1])\n\n\ndef allocate_roles(train, rounds, seed):\n    rng = np.random.default_rng(seed)\n    roles = {f"stage{i+1}": [] for i in range(rounds)} | {"calibration": [], "validation": []}\n    for sp in ("hyena", "leopard"):\n        names = np.array(sorted(train.loc[train.species == sp, "identity"].unique()))\n        rng.shuffle(names)\n        n = max(1, int(round(.2*len(names))))\n        roles["validation"].extend(names[:n])\n        roles["calibration"].extend(names[n:2*n])\n        for i, block in enumerate(np.array_split(names[2*n:], rounds)):\n            roles[f"stage{i+1}"].extend(block)\n    sets = [set(v) for v in roles.values()]\n    assert all(not a & c for i, a in enumerate(sets) for c in sets[i+1:])\n    return {k: np.flatnonzero(train.identity.isin(v).to_numpy()) for k, v in roles.items()}\n\n\ndef split_episode(indices, seed, cap=32, link=(.95, 100)):\n    """Unknown identities absent from gallery; known-query duplicate groups kept whole."""\n    rng = np.random.default_rng(seed)\n    gallery, known, unknown = [], [], []\n    for sp in ("hyena", "leopard"):\n        ix = indices[b.SP[indices] == sp]\n        names = np.unique(b.Y[ix])\n        rng.shuffle(names)\n        nu = max(1, int(round(rng.uniform(.2, .35)*len(names))))\n        unseen = set(names[:nu])\n        for name in names:\n            rows = ix[b.Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in unseen:\n                unknown.extend(rows[:cap])\n                continue\n            held = []\n            groups = b.duplicate_groups(rows, link)\n            target = min(max(1, int(round(.35*len(rows)))), cap)\n            for c in rng.permutation(np.unique(groups)):\n                members = rows[groups == c].tolist()\n                if len(held) < target and len(held)+len(members) < len(rows):\n                    held.extend(members)\n            # Never truncate a duplicate group after selecting it.\n            known.extend(held)\n            gallery.extend(i for i in rows if i not in set(held))\n    gi, qi = np.array(gallery, dtype=int), np.array(known+unknown, dtype=int)\n    if not len(known) or not len(unknown):\n        raise ValueError("Episode lacks known/unknown queries; use a larger role or different seed")\n    for sp in ("hyena", "leopard"):\n        if len(np.unique(b.Y[gi[b.SP[gi] == sp]])) < b.K or np.sum(b.SP[qi] == sp) < 6:\n            raise ValueError("Too few identities/queries per species; reduce --rounds or --candidates")\n    assert not set(b.Y[unknown]) & set(b.Y[gi])\n    assert set(b.Y[known]) <= set(b.Y[gi])\n    return gi, qi, np.where(np.isin(qi, unknown), NEW, b.Y[qi])\n\n\ndef frame(gi, qi, truth=None):\n    s = b.build_frames(gi, qi)\n    if truth is not None:\n        lookup = dict(zip(qi, truth))\n        s["truth"] = np.array([lookup[i] for i in s["order"]])\n        s["q"]["unk"] = (s["truth"] == NEW).astype(int)\n        s["c"]["y"] = (s["c"].ident.to_numpy() == s["truth"][s["c"].q.to_numpy()]).astype(int)\n    return s\n\n\ndef through_stack(s, models):\n    for i, m in enumerate(models):\n        r = b.predict(m, s)\n        if i < len(models)-1:\n            s = b.augment(s, r)\n    return s, r\n\n\ndef fusion_features(s, r):\n    """Only inference-available evidence; no identity strings, labels or image indices."""\n    q = s["q"]\n    pc = r["pc"]\n    top = pc.argmax(1)\n    rows = np.arange(len(q))\n    c = s["c"].iloc[rows*b.K+top]\n    ordered = -np.sort(-pc, axis=1)\n    return pd.DataFrame({\n        "detector": r["pn"], "legacy_new": r["pb"],\n        "raw_sum": r["raw"].sum(1), "raw_max": r["raw"].max(1),\n        "pc_max": ordered[:, 0], "pc_gap": ordered[:, 0]-ordered[:, 1],\n        "pc_entropy": -(pc*np.log(np.maximum(pc, 1e-12))).sum(1),\n        "raw_similarity": c.mxr.to_numpy(), "lda_similarity": c.mxl.to_numpy(),\n        "inliers": np.log1p(c.sx.to_numpy()), "sift_support": np.log1p(c.support.to_numpy()),\n        "same_view_similarity": c.same.to_numpy(), "same_view_available": c.same.notna().to_numpy().astype(int),\n        "gallery_support": np.log1p(c.cnt.to_numpy()), "hyena": q.sp.to_numpy(),\n        "raw_margin": q.rm12.to_numpy(), "lda_margin": q.lm12.to_numpy(),\n    })\n\n\ndef probabilities(s, r, fusion):\n    return {"dalvin_rule": r["pb"], "detector_only": r["pn"],\n            "abhay_fusion": fusion.predict_proba(fusion_features(s, r))[:, 1]}\n\n\ndef evaluate(s, r, probs, all_df, output):\n    meta = all_df.iloc[s["order"]].reset_index(drop=True)\n    unknown = s["truth"] == NEW\n    result = pd.DataFrame({"id": meta.id, "identity": meta.identity, "species": meta.species,\n                           "viewpoint": meta.viewpoint, "truth": s["truth"], "unknown": unknown})\n    labels = s["c"].ident.to_numpy().reshape(-1, b.K)\n    result["candidate_hit"] = (labels == s["truth"][:, None]).any(1)\n    reports = {}\n    for name, p in probs.items():\n        preds = b.rank(s, dict(r, pb=p))\n        score = np.array([next((1/(j+1) for j, v in enumerate(row) if v == t), 0.) for row, t in zip(preds, s["truth"])])\n        result[f"{name}_p_new"] = p\n        result[f"{name}_ap5"] = score\n        result[f"{name}_predictions"] = [" ".join(row) for row in preds]\n        reports[name] = {"map5": float(score.mean()), "known_map5": float(score[~unknown].mean()),\n                         "unknown_map5": float(score[unknown].mean()),\n                         "known_new_at1": float(np.mean(preds[~unknown, 0] == NEW)),\n                         "unknown_new_at1": float(np.mean(preds[unknown, 0] == NEW))}\n    result.to_csv(output / "validation_predictions.csv", index=False)\n    slices = []\n    for fields in (["species", "unknown"], ["viewpoint", "unknown"]):\n        for key, group in result.groupby(fields, dropna=False):\n            slices.append({"slice": str(dict(zip(fields, key))), "n": len(group),\n                           **{name: float(group[f"{name}_ap5"].mean()) for name in probs}})\n    pd.DataFrame(slices).to_csv(output / "validation_slices.csv", index=False)\n    return {"n_queries": len(result), "unknown_fraction": float(unknown.mean()),\n            "known_candidate_recall": float(result.loc[~unknown, "candidate_hit"].mean()),\n            "methods": reports, "fusion_delta_map5": reports["abhay_fusion"]["map5"]-reports["dalvin_rule"]["map5"]}\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument("--data", type=Path, required=True)\n    ap.add_argument("--output", type=Path, default=Path("abhay_output"))\n    ap.add_argument("--seed", type=int, default=2026)\n    ap.add_argument("--rounds", type=int, choices=[1, 2, 3], default=3)\n    ap.add_argument("--sims", type=int, default=12, help="simulations per identity stage")\n    ap.add_argument("--calibration-sims", type=int, default=4)\n    ap.add_argument("--neighbours", type=int, default=150)\n    ap.add_argument("--candidates", type=int, default=15)\n    ap.add_argument("--trees", type=int, default=400)\n    ap.add_argument("--workers", type=int, default=2)\n    ap.add_argument("--predict-test", action="store_true", help="after validation, also write all three submissions")\n    args = ap.parse_args()\n    if min(args.sims, args.calibration_sims, args.neighbours, args.workers, args.trees) < 1 or args.candidates < 5:\n        ap.error("counts must be positive and --candidates >= 5")\n    args.output.mkdir(parents=True, exist_ok=True)\n    config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\n    config["model"] = MODEL_NAME\n    config["model_version"] = MODEL_VERSION\n    config["python"] = platform.python_version()\n    config["packages"] = {n: version(n) for n in ("numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "joblib")}\n    (args.output / "config.json").write_text(json.dumps(config, indent=2))\n    np.random.seed(args.seed)\n    cv2.setNumThreads(1)\n    b.K = args.candidates\n    b.PARAMS.update(n_estimators=args.trees, n_jobs=args.workers, random_state=args.seed)\n    train, test, all_df = load_data(args.data)\n    roles = allocate_roles(train, args.rounds, args.seed)\n    manifest = []\n    for name, ix in roles.items():\n        manifest.extend({"id": b.IDS[i], "identity": b.Y[i], "species": b.SP[i], "role": name} for i in ix)\n    pd.DataFrame(manifest).to_csv(args.output / "identity_roles.csv", index=False)\n    cache = SiftCache(args.data, args.output, args.workers, args.neighbours)\n    models = []\n    split_rows = []\n    def episode(ix, seed, role, link):\n        gi, qi, truth = split_episode(ix, seed, link=link)\n        truth_map = dict(zip(qi, truth))\n        gallery_set = set(gi)\n        split_rows.extend({"episode": seed, "role": role, "id": b.IDS[i],\n                           "assignment": "gallery" if i in gallery_set else "unknown_query" if truth_map[i] == NEW else "known_query"}\n                          for i in np.r_[gi, qi])\n        return frame(gi, qi, truth)\n    for stage in range(args.rounds):\n        name = f"stage{stage+1}"\n        ix = roles[name]\n        cache.activate(ix)\n        sims = []\n        for j in range(args.sims):\n            s = episode(ix, args.seed+1000*(stage+1)+j, name, b.REGIMES[j % len(b.REGIMES)])\n            for model in models:\n                r = b.predict(model, s)\n                s = b.augment(s, r)\n            sims.append(s)\n        print(f"Fitting {name}: {len(ix)} images, {len(sims)} simulations", flush=True)\n        models.append(b.fit(sims))\n    cache.activate(roles["calibration"])\n    cx, cy, cw = [], [], []\n    for j in range(args.calibration_sims):\n        s = episode(roles["calibration"], args.seed+10000+j, "calibration", (.95, 100))\n        s, r = through_stack(s, models)\n        cx.append(fusion_features(s, r))\n        cy.append(s["truth"] == NEW)\n        cw.extend(b.IDS[s["order"]])\n    Xc, yc = pd.concat(cx, ignore_index=True), np.concatenate(cy).astype(int)\n    # An image can recur in calibration episodes: total weight per image is one.\n    counts = pd.Series(cw).value_counts()\n    weights = np.array([1/counts[i] for i in cw])\n    fusion = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),\n                           LogisticRegression(C=1., max_iter=3000, random_state=args.seed))\n    fusion.fit(Xc, yc, logisticregression__sample_weight=weights)\n    feature_names = fusion[0].get_feature_names_out(Xc.columns)\n    pd.DataFrame({"feature": feature_names, "standardized_coefficient": fusion[-1].coef_[0]}).to_csv(\n        args.output / "fusion_coefficients.csv", index=False)\n    pd.DataFrame({"id": cw, "unknown": yc, "sample_weight": weights}).to_csv(args.output / "calibration_examples.csv", index=False)\n    # Save frozen models before final validation. Validation cannot affect fitting.\n    joblib.dump({"models": models, "fusion": fusion, "config": config, "fusion_columns": list(Xc)}, args.output / "models.joblib")\n    cache.activate(roles["validation"])\n    s = episode(roles["validation"], args.seed+20000, "validation", (.95, 100))\n    s, r = through_stack(s, models)\n    report = evaluate(s, r, probabilities(s, r, fusion), all_df, args.output)\n    report["config"] = config\n    report["protocol"] = "identity-disjoint sequential stacking stages, calibration, outer validation"\n    (args.output / "metrics.json").write_text(json.dumps(report, indent=2))\n    pd.DataFrame(split_rows).to_csv(args.output / "episode_splits.csv", index=False)\n    print(json.dumps(report, indent=2), flush=True)\n    if args.predict_test:\n        # Evaluation is complete; only now expose competition test queries.\n        cache.activate(np.arange(len(all_df)))\n        s, r = through_stack(frame(np.arange(len(train)), np.arange(len(train), len(all_df))), models)\n        sample = pd.read_csv(args.data / "sample_submission.csv")\n        for name, p in probabilities(s, r, fusion).items():\n            preds = b.rank(s, dict(r, pb=p))\n            sub = pd.DataFrame({"id": b.IDS[s["order"]], "predictions": [" ".join(v) for v in preds]})\n            sub = sample[["id"]].merge(sub, on="id", how="left", validate="one_to_one")\n            if len(sub) != len(test) or set(sub.id) != set(test.id) or sub.predictions.isna().any():\n                raise ValueError("Submission IDs missing or invalid")\n            known = train.groupby("species").identity.apply(set).to_dict()\n            species = test.set_index("id").species\n            for row in sub.itertuples():\n                tokens = row.predictions.split()\n                assert len(tokens) == len(set(tokens)) == 5\n                assert all(t == NEW or t in known[species[row.id]] for t in tokens)\n            sub.to_csv(args.output / f"submission_{name}.csv", index=False)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
(SCRIPT_DIR / 'README.md').write_text('# Model 5: Open-Set Score Fusion (OSSF)\n\nAuthor contribution: Abhay. Baseline: Dalvin Model 3b (`Models/Dalvin/reid_pipeline.py`).\n\nOSSF learns a logistic combination of existing unknown-animal evidence for the **final placement of `new_individual`**. Retrieval, conditional known-identity ranking and earlier stacking rounds are shared with the baseline. The comparison includes Dalvin\'s legacy rule and the existing detector alone.\n\nThe completed validation run did **not** improve overall MAP@5: baseline **0.916206**, detector-only **0.909947**, OSSF **0.911594**, delta **-0.0046113306982873414**. Fusion improves known-animal MAP@5 slightly but loses more on unseen animals. Dalvin\'s rule remains the strongest method on this split. Full analysis: [RESULTS.md](RESULTS.md). These are validation results, not leaderboard results.\n\n## Method\n\n1. Normalize supplied MIEW-ID descriptors; process each species separately.\n2. Apply PCA (up to 384 dimensions) and shrinkage LDA fitted using episode gallery labels. PCA also sees unlabelled episode queries.\n3. Compute raw/projected similarities, per-identity maximum and top-two scores, centroid similarity, viewpoint evidence and query-neighbour features.\n4. Verify selected same-species neighbours using RootSIFT, ratio testing and partial-affine RANSAC. Retrieve 15 candidate identities using LDA similarity and SIFT evidence.\n5. Train Dalvin\'s unknown detector, known-query candidate scorer and all-query candidate scorer. Apply three sequential stacking stages using neighbour predictions.\n6. Fit median imputation, standardization and logistic regression (`C=1`) on a separate calibration identity group. Inputs include detector and legacy probabilities, candidate-score summaries, margins, SIFT support, viewpoint availability, gallery support and species.\n7. Rank `new_individual` with score `p_new` against known candidates with scores `(1-p_new)*P(identity|known)`. Known identity order is shared across methods, with deterministic tie handling.\n\nThe legacy probability is `(detector_probability + clip(1-sum(binary_candidate_scores), 0, 1))/2`. OSSF tests whether learning this combination improves MAP@5. Logistic loss optimizes binary likelihood; a learned fusion is not guaranteed to improve calibration or ranking. No new image features, raw-image processing or pretrained models are introduced.\n\n## Evaluation protocol\n\nIdentities are shuffled per species with seed 2026. Calibration receives 20%, outer validation 20%, and stack training 60%. The training identities are divided into three non-overlapping stage groups. Stage 1 models produce features for stage 2 identities; stages 1-2 produce features for stage 3 identities. Models from earlier stages have not trained on later-stage identities. This replaces the baseline\'s simulation-level folds to reduce repeated-identity overlap between supervised stages and final evaluation.\n\nWithin each episode, 20-35% of identities are removed from the gallery and labelled `new_individual` for evaluation. Remaining identities supply gallery and known-query sightings where possible. Near-duplicate connected components are held out whole; selected components are never truncated. Singletons and identities with one component may remain gallery-only. Training mixes four duplicate-grouping regimes. Calibration and outer evaluation use cosine >0.95 or >=100 verified SIFT inliers. Unverified pairs limit duplicate detection, so encounter separation is approximate.\n\nThe base stack is frozen before calibration. Repeated calibration sightings receive inverse occurrence weights so each image has total weight one. Fusion is frozen before outer validation. All three methods then use identical validation queries and trained stack outputs. Gallery labels are available to episode retrieval/LDA, as in the competition setup; query truth is excluded from predictor features. Competition test queries are excluded from the validation pipeline.\n\nDefault configuration: three rounds, 12 simulations per stage, four calibration episodes, 150 SIFT neighbours, 15 candidates, 400 candidate-scorer trees and up to 300 detector trees, two CPU workers. The pasted result did not include the complete runtime configuration or library versions; these defaults are not asserted as verified run metadata.\n\nThis protocol uses fewer training identities and smaller galleries per stage than the original full-data baseline. Its scores do not reproduce the team\'s public score or original simulated validation score. Gallery size and unknown prevalence can shift between calibration, evaluation and test. A single 506-query split does not establish statistical superiority. The reported evaluation has now been examined; further tuning requires calibration/development data and a new reserved outer evaluation for final claims.\n\n## Files\n\n| File | Purpose |\n|---|---|\n| `train_eval.py` | Training, final-stage fusion, controlled evaluation and optional submissions |\n| `dalvin_core.py` | Adapted baseline components; attribution retained |\n| `Abhay_OpenSet_Calibration.ipynb` | Self-contained Kaggle notebook |\n| `Abhay_Local_PC.ipynb` | Self-contained Windows notebook |\n| `test_pipeline.py` | Synthetic integration and identity-separation tests |\n| `requirements.txt` | Runtime dependencies |\n| `provenance.json` | Baseline source path, source hash and adaptation list |\n| `RESULTS.md`, `results/` | Measured validation result and analysis |\n| `CONTRIBUTION.md` | Contribution record and verification |\n| `GIT_WORKFLOW.md` | Review, commit and push commands |\n\nNotebook titles and embedded source identify Model 5. Internal method keys (`dalvin_rule`, `detector_only`, `abhay_fusion`) remain stable for compatibility with existing results. Notebook outputs and execution counters are cleared in the repository versions; recorded metrics are stored separately.\n\n## Dataset\n\nCompetition: https://www.kaggle.com/competitions/projectwild3/data\n\n```\n<dataset parent>/\n  reid_student_data_main/\n    train.csv\n    test.csv\n    miewid_ids.csv\n    miewid_features.npy\n    sample_submission.csv\n  sift_features_train.npz/\n  sift_features_test.npz/\n```\n\nSIFT sources may be compressed NPZ archives or extracted directories containing `*_desc.npy` and `*_xy.npy` arrays. The runner searches the metadata directory and its two parent levels. Extracted nested NPY files are supported. Original image folders are unused. Global features are aligned through `miewid_ids.csv`, rather than assumed to follow concatenated CSV order.\n\n## Kaggle execution\n\nImport `Abhay_OpenSet_Calibration.ipynb`, attach the existing `projectwild3` competition input, and execute cells in order. The metadata directory is detected automatically when exactly one complete match exists. `DATA` supports an explicit override. Embedded scripts and outputs are written under `/kaggle/working`; input files remain read-only. A separate code or private data upload is unnecessary.\n\nThe optional install cell targets the active kernel. The optional reduced execution check writes separate artifacts and is not a quality benchmark. Full training uses CPU libraries; selecting a GPU does not automatically accelerate SIFT, PCA/LDA or the configured LightGBM models. Runtime has not been systematically benchmarked.\n\n## Windows execution\n\nThe local notebook defaults to `Path.home()/"Downloads"` for the dataset parent. The project directory is resolved from the notebook working directory or its `Models/Abhay` child and can be overridden. Results are written to timestamped `local_runs` subdirectories.\n\nFrom `Models/Abhay` in PowerShell:\n\n```powershell\npy -3.12 -m venv .venv\n.\\.venv\\Scripts\\python.exe -m pip install -r requirements.txt jupyterlab ipykernel\n.\\.venv\\Scripts\\python.exe -m jupyter lab\n```\n\nCLI equivalent:\n\n```powershell\npython train_eval.py --data "$env:USERPROFILE\\Downloads\\reid_student_data_main" --output ./abhay_output\npython test_pipeline.py -v\n```\n\n`RUN_SMOKE=True` enables a reduced notebook execution check. `PREDICT_TEST=True` adds test CSV generation after validation. Neither switch changes the default full experiment unless enabled. A new timestamped local run starts a new cache; the same output directory is required for cache reuse. Interrupted model-stage training is not resumed automatically.\n\n## Runtime artifacts\n\n- `metrics.json`: MAP@5, known/unknown scores, rank-one new rates, candidate recall, delta, model name, configuration and library versions.\n- `validation_predictions.csv`, `validation_slices.csv`: per-query scores, predictions and grouped error analysis.\n- `fusion_coefficients.csv`: standardized coefficients; correlated features limit causal interpretation.\n- `identity_roles.csv`, `episode_splits.csv`: reproducible role and episode assignments.\n- `calibration_examples.csv`: targets and inverse-occurrence weights.\n- `models.joblib`: frozen stack, fusion and feature schema.\n- `sift_*.npz`: atomic pair cache keyed by IDs and normalized global features. Only role-local requested pairs are exposed during each phase.\n\nOptional test inference keeps the learned weights frozen and uses the full labelled gallery. It exports `submission_dalvin_rule.csv`, `submission_detector_only.csv` and `submission_abhay_fusion.csv`, validating sample IDs/order, five distinct labels and species consistency. It does not submit to Kaggle or automatically declare a winning method. Full-gallery distribution shift remains a limitation.\n\n## Verification and attribution\n\nSynthetic tests cover feature-ID permutation, compressed and extracted SIFT layouts, empty descriptors, actual LightGBM training through three stacking stages, calibration, independent roles, preserved known-identity ordering and valid test submissions. Both notebooks\' code cells compile and embedded modules match the standalone versions. The local data preflight verified 7,163 training sightings, 2,791 test sightings, a 9,954 x 2,152 feature matrix and both SIFT sources.\n\nDalvin\'s source attribution is retained in code and `provenance.json`. Adaptations include import-safe data loading; deterministic seeds and thread settings; bounded PCA dimension; guarded normalization; portable SIFT thread execution with locked ZIP reads; ID alignment; stable score ties; independent evaluation roles; final-stage fusion; and diagnostics. The original shortlist and zero treatment of unverified SIFT pairs are retained. No algorithm change is introduced by documenting the measured result.\n', encoding="utf-8")
(SCRIPT_DIR / 'RESULTS.md').write_text("# Model 5: Open-Set Score Fusion (OSSF) â€” validation analysis\n\nRecorded 24 November 2026 from the completed notebook output. This is an open-set validation result, not a public/private Kaggle leaderboard result. The exact run environment and notebook version were not supplied with the pasted table; `results/reported_metrics.json` preserves that provenance and the displayed precision. Repository defaults are a reproducibility reference, not a verified record of this execution.\n\n## Comparison\n\n| Method | Overall MAP@5 | Known MAP@5 | Unknown MAP@5 | Known incorrectly new at rank 1 | Unknown correctly new at rank 1 |\n|---|---:|---:|---:|---:|---:|\n| Dalvin rule | **0.916206** | 0.887106 | **0.980892** | 0.128940 | **0.961783** |\n| Detector only | 0.909947 | 0.889971 | 0.954352 | 0.114613 | 0.910828 |\n| Model 5 / learned fusion | 0.911594 | **0.890449** | 0.958599 | **0.106017** | 0.923567 |\n\nEvaluation contains 506 queries: 349 known and 157 unknown (31.0277%). Candidate recall for known queries is 0.9799426934, equivalent to 342 of 349 correct identities present in the shortlist. All three methods use the same stack and candidate identity ordering. Differences arise from the position of `new_individual`.\n\nFusion minus baseline MAP@5 is **-0.0046113306982873414**, or about **-0.4611 percentage points**. The exact delta was printed by the notebook; table metrics are rounded to six decimals. Known-query MAP@5 increases by about 0.003343 while unknown-query MAP@5 falls by about 0.022293. Known-animal queries incorrectly assigned new at rank one decrease from 45 to 37. Unknown queries correctly assigned new at rank one decrease from 151 to 145. A reduction in known-new errors is not necessarily an increase in correct identity top-one predictions.\n\n## Species analysis\n\n| Species / query type | n | Dalvin rule | Detector only | Model 5 |\n|---|---:|---:|---:|---:|\n| Hyena / known | 158 | 0.903059 | **0.906224** | 0.900949 |\n| Hyena / unknown | 79 | **0.974684** | 0.940928 | 0.959916 |\n| Leopard / known | 191 | 0.873909 | 0.876527 | **0.881763** |\n| Leopard / unknown | 78 | **0.987179** | 0.967949 | 0.957265 |\n\nKnown-leopard MAP@5 improves, whereas known-hyena MAP@5 decreases slightly. Unknown performance decreases for both species. Several viewpoint slices contain only 1-6 queries, so they do not support reliable viewpoint-specific tuning.\n\n## Interpretation\n\nThe hypothesis was that learning the final combination of unknown evidence would improve ranking over the fixed average. This run does not support that hypothesis overall. Fusion reduces some false unknown decisions for known animals but gives up more reciprocal-rank credit on truly unseen animals. The legacy heuristic's lack of a direct probabilistic interpretation does not imply that it is ineffective for MAP@5.\n\nThe coefficient table shows strong positive weights for the existing detector and legacy unknown score; correlated inputs prevent interpreting coefficient magnitudes as causal importance. Logistic loss optimizes binary likelihood rather than MAP@5, and a pooled linear fusion model may not generalize across species, gallery size, or unknown prevalence. These are possible explanations, not established causes from this table alone.\n\nThe identity-disjoint sequential-stage protocol also differs from Dalvin's original full-data simulation stacking. Its 0.916206 baseline cannot be compared directly with the logged 0.85078 public leaderboard result. No significance claim or leaderboard gain is made from a single split.\n\n## Decision and follow-up\n\nDalvin's rule remains the strongest method on this evaluation. Model 5 is retained as a documented ablation and negative result. A subsequent experiment could constrain changes to the baseline probability through conservative blending or a simpler calibration model. Blend weights and hyperparameters would be selected using calibration/development data only. This examined validation set should not be repeatedly optimized against and described as untouched; a new reserved outer evaluation is required for a final improvement claim.\n", encoding="utf-8")
(SCRIPT_DIR / 'provenance.json').write_text('{\n  "source": "Models/Dalvin/reid_pipeline.py",\n  "source_sha256": "7e4f6be8981ef03955610014cd377d78ccdd3bff3e5d89864a4e466f312d0806",\n  "author": "Dalvin; adapted for Abhay calibration experiment",\n  "changes": [\n    "Removed import-time data loading and original main loop",\n    "Bound PCA components by feature dimension; fixed PCA seed",\n    "Guarded zero-vector normalization",\n    "Explicit deterministic LightGBM parameters; configurable threads and trees via runner",\n    "Original prediction, feature and stacking functions retained; final tie handling specified separately",\n    "Stable final ranking ties preserve conditional identity order for every comparison method"\n  ],\n  "model": "Model 5: Open-Set Score Fusion (OSSF)",\n  "version": "1.0"\n}\n', encoding="utf-8")
(SCRIPT_DIR / 'requirements.txt').write_text('numpy>=1.26,<2.5\npandas>=2.1,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\nlightgbm>=4.3,<5\nopencv-python-headless>=4.8,<5\njoblib>=1.3,<2\n', encoding="utf-8")
print("Model 5 source directory:", SCRIPT_DIR)


## 2. Dataset and configuration
Default configuration: seed 2026; three stacking stages; 12 simulations per stage; four calibration episodes; 150 SIFT neighbours; 15 candidates; 400 candidate-scorer trees; detector capped at 300 trees; two CPU workers. These are repository defaults rather than verified metadata for the recorded validation table.

The projectwild3 competition must be attached as a notebook input. The metadata directory is detected under /kaggle/input; SIFT sources are resolved in that directory or its parent levels. Input files remain read-only.


In [ ]:
import sys, subprocess, json
import pandas as pd

DATA = None  # Or Path("/kaggle/input/competitions/projectwild3/reid_student_data_main")
required = ["train.csv", "test.csv", "miewid_ids.csv", "miewid_features.npy",
            "sample_submission.csv"]
if DATA is None:
    print("Mounted inputs:", [p.name for p in Path("/kaggle/input").iterdir()])
    matches = [p.parent for p in Path("/kaggle/input").rglob("miewid_features.npy")
               if all((p.parent / f).is_file() for f in required)]
    if len(matches) != 1:
        raise ValueError(f"Found {len(matches)} core-data directories. Attach the projectwild3 competition input; set DATA explicitly if multiple copies are mounted: {matches}")
    DATA = matches[0]
DATA = Path(DATA)
assert all((DATA / f).is_file() for f in required), "Missing competition files"
OUTPUT = Path("/kaggle/working/abhay_output")
SEED, ROUNDS, SIMS, CALIBRATION_SIMS = 2026, 3, 12, 4
NEIGHBOURS, CANDIDATES, TREES, WORKERS = 150, 15, 400, 2
PREDICT_TEST = False
print("Dataset:", DATA)
print("Outputs:", OUTPUT)


## 3. Reduced execution check
The optional reduced run verifies execution with fewer simulations, trees and SIFT neighbours. Its output directory is separate from the full experiment. Reduced-run scores are not a model-quality benchmark.


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run([sys.executable, str(SCRIPT_DIR / "train_eval.py"),
                    "--data", str(DATA), "--output", "/kaggle/working/abhay_smoke",
                    "--sims", "1", "--calibration-sims", "1", "--neighbours", "10",
                    "--trees", "20", "--workers", str(WORKERS)], check=True)


## 4. Training, calibration and final evaluation
The subprocess uses the active kernel's interpreter. Sequential stacking models are fitted first, logistic fusion second and outer evaluation last. No hyperparameter selection uses the outer evaluation. Optional competition-test inference runs only after evaluation and uses frozen weights with the full labelled gallery.

SIFT matching and repeated PCA/LDA fitting are CPU-intensive. Pair caches are reusable within the same output directory; interrupted model-stage training is not resumed. A new local timestamped run creates a new cache.


In [ ]:
command = [sys.executable, str(SCRIPT_DIR / "train_eval.py"),
           "--data", str(DATA), "--output", str(OUTPUT), "--seed", str(SEED),
           "--rounds", str(ROUNDS), "--sims", str(SIMS),
           "--calibration-sims", str(CALIBRATION_SIMS), "--neighbours", str(NEIGHBOURS),
           "--candidates", str(CANDIDATES), "--trees", str(TREES), "--workers", str(WORKERS)]
if PREDICT_TEST:
    command.append("--predict-test")
print(" ".join(command))
subprocess.run(command, check=True)


## 5. Current run metrics and error analysis
MAP@5 is mean reciprocal rank through position five. Known-new-at-one measures known queries incorrectly placed behind new at rank one; unknown-new-at-one measures correct rank-one rejection of unseen identities. Candidate recall applies only to known queries.

The following cells display current-run metrics, species/viewpoint slices, largest per-query losses/gains and standardized fusion coefficients. Correlated inputs limit causal interpretation of coefficient magnitudes. The historical result is documented separately below.


In [ ]:
report = json.loads((OUTPUT / "metrics.json").read_text())
comparison = pd.DataFrame(report["methods"]).T.rename(index={"dalvin_rule": "Dalvin rule", "detector_only": "Detector only", "abhay_fusion": "Model 5: OSSF"})
display(comparison)
print("Fusion minus baseline MAP@5:", report["fusion_delta_map5"])
print("Queries:", report["n_queries"], "Unknown fraction:", report["unknown_fraction"])
print("Known candidate recall:", report["known_candidate_recall"])
display(pd.read_csv(OUTPUT / "validation_slices.csv"))
errors = pd.read_csv(OUTPUT / "validation_predictions.csv")
errors["delta_ap5"] = errors.abhay_fusion_ap5 - errors.dalvin_rule_ap5
display(errors.sort_values("delta_ap5").head(15))
display(errors.sort_values("delta_ap5", ascending=False).head(15))
display(pd.read_csv(OUTPUT / "fusion_coefficients.csv"))


## Recorded validation analysis — 24 November 2026

| Method | MAP@5 | Known MAP@5 | Unknown MAP@5 |
|---|---:|---:|---:|
| Dalvin rule | **0.916206** | 0.887106 | **0.980892** |
| Detector only | 0.909947 | 0.889971 | 0.954352 |
| Model 5: OSSF | 0.911594 | **0.890449** | 0.958599 |

The 506 queries comprise 349 known and 157 unknown sightings. Known candidate recall is 342/349 (97.9943%). Known queries incorrectly ranked new at position one fall from 45 to 37, but unknown queries correctly ranked new fall from 151 to 145. Known-leopard MAP@5 improves; known-hyena MAP@5 decreases slightly. The total loss is about 0.4611 percentage points of MAP@5.

Conclusion: the fixed heuristic remains strongest on this split. Learned fusion is retained as a controlled ablation and negative result. Small viewpoint slices and a single evaluation split do not establish general superiority. Further changes require calibration/development selection and an additional reserved outer evaluation. The supplied output did not include the exact notebook version or full environment/configuration; reference defaults are documented separately.


## 6. Result archive
The archive contains runtime configuration, models, split manifests, metrics and predictions. Runtime artifacts are excluded from Git. Source notebooks and the small reported-results summary are versioned separately. Test CSVs are generated only when PREDICT_TEST is enabled before training; no automatic Kaggle submission occurs.


In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/abhay_experiment", "zip", OUTPUT)
print("Download:", archive)
print("Full documentation:", SCRIPT_DIR / "README.md")
print("Available submissions:", [p.name for p in OUTPUT.glob("submission_*.csv")])
